In [70]:
import numpy as np
import torch
from sklearn.metrics.pairwise import cosine_distances

from mirroreval.hf_utilities import get_hf_pipeline, get_hf_model, get_hf_tokenizer

In [71]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [72]:
model = get_hf_model("FacebookAI/roberta-large").to(device)
tokenizer = get_hf_tokenizer("FacebookAI/roberta-large")

Some weights of RobertaModel were not initialized from the model checkpoint at FacebookAI/roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [73]:
def self_avgcosine(sentences_groups):
    # CREDIT: github.com/LivNLP/Evaluating-Diversity-Metrics

    scores = []
    for sentences in sentences_groups:
        inputs = tokenizer(
            sentences, padding=True, truncation=True, return_tensors="pt"
        )
        inputs = {
            key: value.to(device) for key, value in inputs.items()
        }  # Move input data to GPU

        # Get the embeddings
        with torch.no_grad():
            embeddings = model(
                **inputs, output_hidden_states=True, return_dict=True
            ).pooler_output

        # Calculate cosine similarities and sum them up for each sentence
        embeddings_norm = embeddings / embeddings.norm(dim=1, keepdim=True)

        # Compute similarity matrix
        similarity_matrix = torch.matmul(embeddings_norm, embeddings_norm.T)

        # Exclude self-comparisons
        mask = torch.ones_like(similarity_matrix) - torch.eye(
            len(embeddings), device=embeddings.device
        )
        total_similarities = (similarity_matrix * mask).sum().item()
        num_comparisons = mask.sum().item()

        # Calculate average similarity
        average_similarity = total_similarities / num_comparisons
        scores.append(average_similarity)
    return np.mean(scores)

In [74]:
def chamfer_distance(sentences_groups):
    # CREDIT: github.com/LivNLP/Evaluating-Diversity-Metrics

    scores = []
    for sentences in sentences_groups:
        inputs = tokenizer(
            sentences, padding=True, truncation=True, return_tensors="pt"
        )
        inputs = {
            key: value.to(device) for key, value in inputs.items()
        }  # Move input data to GPU

        # Get the embeddings
        with torch.no_grad():
            embeddings = model(
                **inputs, output_hidden_states=True, return_dict=True
            ).pooler_output

        # Calculate cosine similarities and sum them up for each sentence
        embeddings_norm = embeddings / embeddings.norm(dim=1, keepdim=True)
        embeddings_np = embeddings_norm.cpu().numpy()
        cosine_dist_matrix = cosine_distances(embeddings_np)
        np.fill_diagonal(cosine_dist_matrix, np.inf)
        min_distances = np.min(cosine_dist_matrix, axis=1)
        chamfer_distance_value = np.mean(min_distances)

        scores.append(chamfer_distance_value)
    return np.mean(scores)

In [75]:
sentences_groups_low_diversity = [["The goat jumped over ", 
                     "The goat jumped over", 
                     "The goat jumped over", 
                     "The goat jumped over", ]]
# sentences_groups_high_diversity = [["The goat jumped over the dog to feed it with a bottle.", 
#                      "To go on vacation, the dog fed the goat by jumping into a bottle.", 
#                      "A hungry goat said that it will jump into a bottle and feed itself like a dog.", 
#                      "The dog took a deep breath and jumped inside the feeding bottle while the goat watched."]]
# sentences_groups_high_diversity = [["A swarm of parrotfish drifted over the coral ridge, their scales glinting like shards of turquoise glass in the sunlight.", 
#                      "The last transmission from the lunar outpost crackled through the static, carrying only the sound of hollow wind where oxygen once hissed.", 
#                      "Vendors shouted in Thai as the scent of chili and lemongrass spilled from sizzling woks under strings of flickering neon lights.", 
#                      "I tightened the straps of my frost-bitten pack and wrote, with a trembling hand, that the mountain did not forgive but it remembered."]]
sentences_groups_high_diversity = [["A swarm of parrotfish drifted over the coral ridge, their scales glinting like shards of turquoise glass in the sunlight.", 
                     "WAKAKA KAKAKFNIUJDHN DNISND OISND ", ]]

In [76]:
chamfer_low_diversity = chamfer_distance(sentences_groups_low_diversity)
chamfer_high_diversity = chamfer_distance(sentences_groups_high_diversity)

In [77]:
print(chamfer_low_diversity)
print(chamfer_high_diversity)

0.00011022389
0.04488331


In [78]:
self_cosine_low_diversity = self_avgcosine(sentences_groups_low_diversity)
self_cosine_high_diversity = self_avgcosine(sentences_groups_high_diversity)

In [79]:
print(self_cosine_low_diversity)
print(self_cosine_high_diversity)

0.9997797012329102
0.9551167488098145


Run it on the data

In [83]:
import pandas as pd

In [89]:
previous_results = "/home/jayo/repos/MIRROR-Eval/results/results.jsonl"

In [90]:
df = pd.read_json(previous_results, lines=True)

In [91]:
# Remove model_name and prompt columns, then remove duplicate rows.
# The rows have lists in them, so we can't do a simple drop_duplicates.
df_reduced = df.drop(columns=['model_name', 'prompt', 'output_both_sets', 'output_set_1', 'output_set_2'])
df_reduced = df_reduced.drop_duplicates(subset=['src'])

In [92]:
df_reduced.head()

,split_name,src,set1,set2,set1_label,set2_label,Quality_Set1,Quality_Set2,Diversity_Set1,Diversity_Set2,llm_quality,llm_diversity
0,semeval_evaluation,We need bread on rainy days.,[Rainy days do not have any specific relation ...,[Rainy days don't have any correlation with th...,diversified,icd,4.6,4.0,3.0,3.0,0,2
4,semeval_evaluation,She drove her car to space.,"[Space travel requires specialized vehicles, n...",[It is not currently possible for cars to trav...,icd,default,5.0,4.6,4.4,2.2,0,0
8,semeval_evaluation,He deposited drugs into the bank.,[Drug deposits are a criminal offense and woul...,[Banks are not meant for storing illegal subst...,icd,diversified,4.6,4.6,3.0,3.0,2,2
12,semeval_evaluation,I'm dirty now so I need to have lunch.,[Being dirty does not necessarily have any rel...,[The level of cleanliness does not determine o...,default,icd,4.8,4.8,2.4,3.6,2,1
16,semeval_evaluation,Programmers usually don't use computers.,[Programmers need computers to write and test ...,[Programmers need computers to write and test ...,default,icd,4.6,5.0,2.6,3.0,1,1


In [93]:
print(df_reduced.shape)

(1920, 12)


In [94]:
# Iterate through each line in the previous_results jsonl file
import json
from tqdm import tqdm
with open(previous_results, "r") as f:
    for idx, line in enumerate(tqdm(f)):
        data = json.loads(line)
        set1 = data["set1"]
        set2 = data["set2"]

        chamfer_distance_set1 = chamfer_distance([set1])
        chamfer_distance_set2 = chamfer_distance([set2])

        data["chamfer_distance_set1"] = chamfer_distance_set1.item()
        data["chamfer_distance_set2"] = chamfer_distance_set2.item()

        if chamfer_distance_set1 > chamfer_distance_set2:
            data["chamfer_distance_output"] = 0
        elif chamfer_distance_set1 < chamfer_distance_set2:
            data["chamfer_distance_output"] = 1
        else:
            print('tie!', chamfer_distance_set1, chamfer_distance_set2)
            data["chamfer_distance_output"] = 2

        self_avgcosine_set1 = self_avgcosine([set1])
        self_avgcosine_set2 = self_avgcosine([set2])

        data["self_avgcosine_set1"] = self_avgcosine_set1.item()
        data["self_avgcosine_set2"] = self_avgcosine_set2.item()

        if self_avgcosine_set1 < self_avgcosine_set2:
            data["self_avgcosine_output"] = 0
        elif self_avgcosine_set1 > self_avgcosine_set2:
            data["self_avgcosine_output"] = 1
        else:
            print('tie!', self_avgcosine_set1, self_avgcosine_set2)
            data["self_avgcosine_output"] = 2

        # Save out to json file
        with open("/home/jayo/repos/MIRROR-Eval/results/chamfer_cossim.jsonl", "a") as out_f:
            out_f.write(json.dumps(data) + "\n")


73it [00:10,  5.90it/s]

tie! 0.0023804505 0.0023804505
tie! 0.9963658650716146 0.9963658650716146
tie! 0.0023804505 0.0023804505


74it [00:10,  5.56it/s]

tie! 0.9963658650716146 0.9963658650716146
tie! 0.0023804505 0.0023804505


75it [00:10,  5.26it/s]

...

20677it [50:37,  7.17it/s]

tie! 0.0024962823 0.0024962823
tie! 0.0024962823 0.0024962823


20679it [50:38,  7.01it/s]

tie! 0.0024962823 0.0024962823
tie! 0.0024962823 0.0024962823


23040it [56:31,  6.79it/s]
